# K-TIDE — LoRA SFT on Colab (T4)

Trains one condition (e.g. M1), merges and packages the model for the evaluation notebook, and
generates answers for the evaluation set under the official conditions (CUDA FP16, greedy).

**Before running:** put the private bundle `ktide_bundle_<tag>.zip` (made locally with
`python -m ktide.colab_bundle`) in your Google Drive folder `K-TIDE/`. Runtime → GPU (T4).
This notebook contains no private data and is safe in the public repo.

In [ ]:
# ==== settings ====
BUNDLE = "ktide_bundle_m1.zip"            # file name in MyDrive/K-TIDE/
QA = ["data_private/train/m1_lecture_only/qa.jsonl"]
VARIANT = "translated"                   # translated (M1, M3) | preserved (M2, M4)
CONDITION = "M1"
EPOCHS, LR, R, ALPHA, TARGETS = 3, 1e-4, 16, 32, "all-linear"
SAVE_MODEL_ZIP_TO_DRIVE = False          # True only for the final assignment model (≈3 GB)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, subprocess, sys, zipfile
DRIVE = "/content/drive/MyDrive/K-TIDE"
WORK = "/content/ktide"
if not os.path.isdir(WORK):
    os.makedirs(WORK)
    zipfile.ZipFile(f"{DRIVE}/{BUNDLE}").extractall(WORK)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# Same pinned versions as the evaluation notebook (+ peft for LoRA). torch = Colab preinstalled.
os.environ["TOKENIZERS_PARALLELISM"] = "false"; os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
    "transformers==4.57.1", "accelerate==1.11.0", "huggingface_hub==0.36.0", "safetensors==0.6.2",
    "peft==0.17.1", "pydantic>=2.9"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", WORK], check=True)
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
def run(*args):
    r = subprocess.run([sys.executable, "-m", *args], cwd=WORK, capture_output=True, text=True)
    print(r.stdout[-4000:]); print(r.stderr[-4000:])
    r.check_returncode()

# 1. token-level loss-masking check on the real data
run("ktide.train.data", "--qa", *QA, "--variant", VARIANT, "--check")

In [ ]:
# 2. LoRA SFT
OUT = f"/content/outputs/{CONDITION}"
run("ktide.train.sft", "--qa", *QA, "--variant", VARIANT, "--condition", CONDITION, "--out", OUT,
    "--epochs", str(EPOCHS), "--lr", str(LR), "--r", str(R), "--alpha", str(ALPHA), "--targets", TARGETS)

In [ ]:
# 3. merge + package + verify (professor unpack_model, greedy decoding, size)
run("ktide.train.export", "--adapter", f"{OUT}/adapter", "--out", f"{OUT}/package")

In [ ]:
# 4. evaluation answers under official conditions (professor load_model + answer_questions, CUDA FP16)
run("ktide.generate", "--experiment", "main", "--condition", CONDITION, "--model-dir", f"{OUT}/package/qwen25_sft_model",
    "--questions-file", "data_private/evalset/eval_v1_prefreeze_questions.json", "--tag", "colab-t4",
    "--notes", f"{CONDITION} LoRA, variant={VARIANT}")

In [ ]:
# 5. save results to Drive (run dir + adapter + configs; the model ZIP only if requested)
import glob, json
run_dir = sorted(glob.glob(f"{WORK}/runs/main/{CONDITION}/*colab-t4*"))[-1]
res = f"/content/{CONDITION}_results"
shutil.rmtree(res, ignore_errors=True)
shutil.copytree(run_dir, f"{res}/run/{os.path.basename(run_dir)}")
shutil.copytree(f"{OUT}/adapter", f"{res}/adapter")
for f in ("train_config.json", "train_log.json"): shutil.copy(f"{OUT}/{f}", res)
shutil.copy(f"{OUT}/package/export_report.json", res)
archive = shutil.make_archive(f"/content/{CONDITION}_results", "zip", res)
shutil.copy(archive, DRIVE)
if SAVE_MODEL_ZIP_TO_DRIVE:
    shutil.copy(f"{OUT}/package/qwen25_sft_model.zip", f"{DRIVE}/{CONDITION}_qwen25_sft_model.zip")
print("saved:", archive, "->", DRIVE)
print(json.dumps(json.load(open(f"{OUT}/train_config.json")), indent=1)[:1500])